In [ ]:
# h2o_ai/health_advisor.py

import requests
import concurrent.futures
from typing import Dict, Optional

# ============================================
# LOCATION SERVICE (Multi-API Consensus)
# ============================================
class LocationService:
    """
    Gets location using multiple IP geolocation APIs.
    No browser needed. No API keys required.
    """
    
    def __init__(self):
        self.cached_location = None
    
    def get_location(self) -> Dict:
        """Get current location with consensus from multiple APIs"""
        if self.cached_location:
            return self.cached_location
        
        location = self._get_consensus_location()
        
        if location:
            self.cached_location = location
            return location
        
        return self._get_fallback_location()
    
    def _get_consensus_location(self) -> Optional[Dict]:
        """Query multiple APIs in parallel, return most common result"""
        
        apis = [
            self._try_ipapi,
            self._try_ipinfo,
            self._try_ipapi_co
        ]
        
        # Try geocoder if available
        try:
            import geocoder
            apis.append(self._try_geocoder)
        except ImportError:
            pass
        
        results = []
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(api) for api in apis]
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result and result.get('city') != 'Unknown':
                        results.append(result)
                except:
                    pass
        
        if not results:
            return None
        
        # Find most common city
        city_votes = {}
        for r in results:
            city = r['city']
            city_votes[city] = city_votes.get(city, 0) + 1
        
        best_city = max(city_votes, key=city_votes.get)
        confidence = f"{city_votes[best_city]}/{len(results)} APIs agreed"
        
        # Return result with most votes
        for r in results:
            if r['city'] == best_city:
                r['confidence'] = confidence
                return r
        
        return results[0]
    
    def _try_ipapi(self) -> Optional[Dict]:
        """ip-api.com - Most reliable free API"""
        try:
            response = requests.get(
                'http://ip-api.com/json/',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if data.get('status') == 'success':
                return {
                    'lat': data.get('lat'),
                    'lon': data.get('lon'),
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('regionName', ''),
                    'country': data.get('country', 'Unknown'),
                    'ip': data.get('query', ''),
                    'timezone': data.get('timezone', ''),
                    'source': 'ip-api'
                }
        except Exception as e:
            print(f"   ⚠️  ip-api failed: {e}")
        return None
    
    def _try_ipinfo(self) -> Optional[Dict]:
        """ipinfo.io"""
        try:
            response = requests.get(
                'https://ipinfo.io/json',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if 'loc' in data:
                lat, lon = data['loc'].split(',')
                return {
                    'lat': float(lat),
                    'lon': float(lon),
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('region', ''),
                    'country': data.get('country', 'Unknown'),
                    'ip': data.get('ip', ''),
                    'timezone': data.get('timezone', ''),
                    'source': 'ipinfo'
                }
        except Exception as e:
            print(f"   ⚠️  ipinfo failed: {e}")
        return None
    
    def _try_geocoder(self) -> Optional[Dict]:
        """Geocoder library"""
        try:
            import geocoder
            g = geocoder.ip('me')
            
            if g.ok and g.city:
                return {
                    'lat': g.latlng[0] if g.latlng else None,
                    'lon': g.latlng[1] if g.latlng else None,
                    'city': g.city,
                    'state': g.state or '',
                    'country': g.country or 'Unknown',
                    'ip': g.ip,
                    'source': 'geocoder'
                }
        except Exception as e:
            print(f"   ⚠️  geocoder failed: {e}")
        return None
    
    def _try_ipapi_co(self) -> Optional[Dict]:
        """ipapi.co"""
        try:
            response = requests.get(
                'https://ipapi.co/json/',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if 'error' not in data and data.get('city'):
                return {
                    'lat': data.get('latitude'),
                    'lon': data.get('longitude'),
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('region', ''),
                    'country': data.get('country_name', 'Unknown'),
                    'ip': data.get('ip', ''),
                    'timezone': data.get('timezone', ''),
                    'source': 'ipapi.co'
                }
        except Exception as e:
            print(f"   ⚠️  ipapi.co failed: {e}")
        return None
    
    def _get_fallback_location(self) -> Dict:
        """Default to Mumbai if everything fails"""
        return {
            'lat': 19.0760,
            'lon': 72.8777,
            'city': 'Mumbai',
            'state': 'Maharashtra',
            'country': 'India',
            'ip': 'unknown',
            'timezone': 'Asia/Kolkata',
            'source': 'fallback'
        }


# ============================================
# WEATHER SERVICE - Using Open-Meteo
# ============================================
def get_weather_by_coords(lat, lon):
    """
    Fetch current weather data from Open-Meteo API (free, no API key required).
    Returns a dictionary with temperature, humidity, wind speed, pressure, UV index, etc.
    """
    # Parameters for Open-Meteo API
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,uv_index",
        "timezone": "auto"
    }

    try:
        response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        # Extract the data we need
        current = data.get("current", {})
        weather = {
            "temperature": current.get("temperature_2m"),
            "humidity": current.get("relative_humidity_2m"),
            "wind_speed": current.get("wind_speed_10m"),
            "pressure": current.get("pressure_msl"),
            "uv_index": current.get("uv_index")
        }
        return weather
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error while fetching weather: {e}")
        return None
    except KeyError as e:
        print(f"❌ Unexpected API response format: {e}")
        return None


# ============================================
# HEALTH RECOMMENDATION ENGINE
# ============================================
def classify_weather_type(temperature, humidity):
    """Classify weather based on temperature and humidity."""
    if temperature >= 30:
        heat_category = "hot"
    elif temperature >= 20:
        heat_category = "warm"
    elif temperature >= 10:
        heat_category = "cool"
    else:
        heat_category = "cold"

    if humidity >= 70:
        humidity_category = "humid"
    elif humidity >= 40:
        humidity_category = "normal"
    else:
        humidity_category = "dry"

    return heat_category, humidity_category


def water_intake_recommendation(temperature, humidity):
    """Compute a daily water intake recommendation in litres."""
    # Base intake (average adult)
    base_intake = 2.5

    # Adjust for temperature
    if temperature >= 30:
        temp_adjust = 0.6
    elif temperature >= 20:
        temp_adjust = 0.3
    elif temperature <= 10:
        temp_adjust = -0.2
    else:
        temp_adjust = 0.0

    # Adjust for humidity
    if humidity >= 70:
        humid_adjust = 0.3
    elif humidity >= 40:
        humid_adjust = 0.0
    else:
        humid_adjust = -0.1

    # A very cold + dry combo might still need extra
    if temperature <= 10 and humidity <= 40:
        humid_adjust = 0.2

    recommended = base_intake + temp_adjust + humid_adjust

    return round(recommended, 1)


def food_recommendation(heat_category, humidity_category):
    """Return food suggestions based on weather type."""
    if heat_category == "hot":
        if humidity_category == "humid":
            foods = """
            🍚 Light & easy-to-digest meals (khichdi, steamed veggies, soups)
            🥒 Highly hydrating foods: cucumber, zucchini, bottle gourd, watermelon, oranges
            🥣 Probiotics for gut health: curd, buttermilk, fermented pickles
            🚫 Avoid heavy, oily, spicy dishes that increase body heat"""
        else:  # hot & dry
            foods = """
            🍉 High-water fruits: watermelon, muskmelon, berries, cucumber
            🥗 Light salads, coconut water, fresh fruit juices
            🧉 Electrolyte balance: lassi, buttermilk, lemon water
            🚫 Reduce caffeine and alcohol"""
    elif heat_category == "warm":
        foods = """
        🥗 Normal seasonal foods: include plenty of fresh vegetables
        🥛 Dairy in moderation, light proteins
        🥒 Keep hydrating snacks (cucumber, tomatoes)
        🚫 Avoid excessive fried or processed food"""
    elif heat_category == "cool":
        foods = """
        🍲 Warm, cooked meals (soups, stews, porridge)
        🥕 Root vegetables: carrots, sweet potatoes, beets
        🍎 Seasonal fruits: apples, pears, pomegranates
        🥛 Warm milk with spices (turmeric, ginger)"""
    else:  # cold
        foods = """
        🍲 Hot, nourishing meals (hearty soups, khichdi, broths)
        🥜 Warming nuts & seeds: sesame seeds, almonds, walnuts
        🍚 Whole grains: millets (ragi, bajra), oats
        🧉 Herbal teas: ginger-tulsi, kadha, haldi-doodh
        🚫 Avoid cold, raw, or icy foods"""

    return foods.strip()


def print_recommendations(weather, heat_category, humidity_category):
    """Print the advice in a pretty format."""
    print("\n🌡️ **Weather-Adapted Health Guide**")
    print(f"   Weather condition: {heat_category.capitalize()} & {humidity_category}")

    water = water_intake_recommendation(weather["temperature"], weather["humidity"])
    print(f"   💧 Recommended daily water intake: {water} litres")
    if water > 3.0:
        print("       ☀️ Hot/humid conditions increase water loss – drink little and often")
    elif weather["temperature"] < 15 and weather["humidity"] < 40:
        print("       ❄️ Cool & dry air still requires good hydration – don't skip it")

    print("\n🍽️ **Food suggestions for today:**")
    print(food_recommendation(heat_category, humidity_category))


# ============================================
# MAIN APPLICATION
# ============================================
def main():
    print("\n" + "="*60)
    print("   💧  H2O AI - Intelligent Health & Hydration Advisor  💧")
    print("="*60)
    
    print("\n🔍 Detecting your location...")
    print("   Querying multiple geolocation services...")
    
    location_service = LocationService()
    location = location_service.get_location()
    
    print(f"\n✅ Location detected: {location['city']}, {location['country']}")
    
    print("\n📍 YOUR LOCATION")
    print("-" * 40)
    print(f"   🏙️  City:      {location['city']}")
    print(f"   🏛️  State:     {location.get('state', 'N/A')}")
    print(f"   🌍 Country:   {location['country']}")
    print(f"   📡 IP:        {location.get('ip', 'N/A')}")
    print(f"   🌐 Source:    {location.get('source', 'unknown')}")
    if location.get('confidence'):
        print(f"   🎯 Accuracy:  {location['confidence']}")
    
    print(f"   📍 Latitude:  {location['lat']}")
    print(f"   📍 Longitude: {location['lon']}")

    print("\n🌤️ Fetching current weather data...")
    weather = get_weather_by_coords(location["lat"], location["lon"])
    
    if not weather:
        print("❌ Could not retrieve weather data.")
        return

    print("\n--- Weather Report ---")
    print(f"{weather}")
    print(f"🌡️ Temperature:   {weather['temperature']}°C")
    print(f"💧 Humidity:      {weather['humidity']}%")
    print(f"💨 Wind speed:    {weather['wind_speed']} km/h")
    print(f"🌬️ Pressure:      {weather['pressure']} hPa")
    print(f"☀️ UV Index:      {weather['uv_index']} (higher = stronger sun protection needed)")

    # Classify and give advice
    heat_cat, humid_cat = classify_weather_type(weather["temperature"], weather["humidity"])
    print_recommendations(weather, heat_cat, humid_cat)
    
    print("\n" + "="*60)
    print("   💙 Stay healthy, stay hydrated! - H2O AI")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()


   💧  H2O AI - Intelligent Health & Hydration Advisor  💧

🔍 Detecting your location...
   Querying multiple geolocation services...

✅ Location detected: Coimbatore, India

📍 YOUR LOCATION
----------------------------------------
   🏙️  City:      Coimbatore
   🏛️  State:     Tamil Nadu
   🌍 Country:   India
   📡 IP:        2401:4900:1ce3:9726:1543:5efe:4d5a:3b2a
   🌐 Source:    ipapi.co
   🎯 Accuracy:  2/3 APIs agreed
   📍 Latitude:  11.0102
   📍 Longitude: 76.9701

🌤️ Fetching current weather data...

--- Weather Report ---


KeyError: slice(None, None, None)

Flagged Changes:

1. Showing Pudukottai Instead of real location
2. Need to incorporate streaming option along with more specific water and food recommendations.

In [2]:
# h2o_ai/services/location_service.py

import requests
from typing import Dict, Optional
import concurrent.futures

class LocationService:
    """
    Gets location using IP geolocation APIs.
    No browser needed. Works instantly.
    """
    
    def __init__(self):
        self.cached_location = None
    
    def get_location(self) -> Dict:
        """Get current location"""
        if self.cached_location:
            return self.cached_location
        
        # Try multiple APIs and find consensus
        location = self._get_consensus_location()
        
        if location:
            self.cached_location = location
            return location
        
        return self._get_fallback_location()
    
    def _get_consensus_location(self) -> Optional[Dict]:
        """Query multiple APIs in parallel, return most common result"""
        
        apis = [
            self._try_ipapi,
            self._try_ipinfo,
            self._try_geocoder,
            self._try_ipapi_co
        ]
        
        results = []
        
        # Query all APIs simultaneously
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(api) for api in apis]
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result and result.get('city') != 'Unknown':
                        results.append(result)
                        print(f"✅ Got location from {result['source']}: {result['city']}")
                except:
                    pass
        
        if not results:
            return None
        
        # Find most common city
        city_votes = {}
        for r in results:
            city = r['city']
            city_votes[city] = city_votes.get(city, 0) + 1
        
        best_city = max(city_votes, key=city_votes.get)
        
        # Return result with most votes
        for r in results:
            if r['city'] == best_city:
                r['confidence'] = f"{city_votes[best_city]}/{len(results)} APIs agreed"
                return r
        
        return results[0]
    
    def _try_ipapi(self) -> Optional[Dict]:
        """ip-api.com - Most reliable free API"""
        try:
            response = requests.get(
                'http://ip-api.com/json/',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if data.get('status') == 'success':
                return {
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('regionName', ''),
                    'country': data.get('country', ''),
                    'latitude': data.get('lat'),
                    'longitude': data.get('lon'),
                    'timezone': data.get('timezone', ''),
                    'source': 'ip-api'
                }
        except Exception as e:
            print(f"ip-api failed: {e}")
        return None
    
    def _try_ipinfo(self) -> Optional[Dict]:
        """ipinfo.io - Good accuracy"""
        try:
            response = requests.get(
                'https://ipinfo.io/json',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if 'loc' in data:
                lat, lon = data['loc'].split(',')
                return {
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('region', ''),
                    'country': data.get('country', ''),
                    'latitude': float(lat),
                    'longitude': float(lon),
                    'timezone': data.get('timezone', ''),
                    'source': 'ipinfo'
                }
        except Exception as e:
            print(f"ipinfo failed: {e}")
        return None
    
    def _try_geocoder(self) -> Optional[Dict]:
        """Geocoder library - Built-in Python package"""
        try:
            import geocoder
            g = geocoder.ip('me')
            
            if g.ok and g.city:
                return {
                    'city': g.city,
                    'state': g.state or '',
                    'country': g.country or '',
                    'latitude': g.latlng[0] if g.latlng else None,
                    'longitude': g.latlng[1] if g.latlng else None,
                    'source': 'geocoder'
                }
        except Exception as e:
            print(f"geocoder failed: {e}")
        return None
    
    def _try_ipapi_co(self) -> Optional[Dict]:
        """ipapi.co - Another reliable API"""
        try:
            response = requests.get(
                'https://ipapi.co/json/',
                timeout=5,
                headers={'User-Agent': 'H2O-AI/1.0'}
            )
            data = response.json()
            
            if 'error' not in data:
                return {
                    'city': data.get('city', 'Unknown'),
                    'state': data.get('region', ''),
                    'country': data.get('country_name', ''),
                    'latitude': data.get('latitude'),
                    'longitude': data.get('longitude'),
                    'timezone': data.get('timezone', ''),
                    'source': 'ipapi.co'
                }
        except Exception as e:
            print(f"ipapi.co failed: {e}")
        return None
    
    def _get_fallback_location(self) -> Dict:
        """Default to Mumbai if everything fails"""
        return {
            'city': 'Mumbai',
            'state': 'Maharashtra',
            'country': 'India',
            'latitude': 19.0760,
            'longitude': 72.8777,
            'source': 'fallback'
        }


# ============================================
# TEST IT RIGHT NOW
# ============================================
if __name__ == "__main__":
    print("\n" + "="*50)
    print("📍 TESTING LOCATION DETECTION")
    print("="*50 + "\n")
    
    service = LocationService()
    location = service.get_location()
    
    print("\n" + "="*50)
    print("✅ YOUR CURRENT LOCATION")
    print("="*50)
    print(f"🏙️  City:      {location['city']}")
    print(f"🏛️  State:     {location.get('state', 'N/A')}")
    print(f"🌍 Country:   {location['country']}")
    print(f"📍 Latitude:  {location['latitude']}")
    print(f"📍 Longitude: {location['longitude']}")
    if 'confidence' in location:
        print(f"📊 Confidence: {location['confidence']}")
    print(f"🔍 Source:     {location.get('source', 'unknown')}")
    print("="*50)


📍 TESTING LOCATION DETECTION

geocoder failed: No module named 'geocoder'
✅ Got location from ip-api: Pudukkottai
✅ Got location from ipapi.co: Coimbatore
✅ Got location from ipinfo: Coimbatore

✅ YOUR CURRENT LOCATION
🏙️  City:      Coimbatore
🏛️  State:     Tamil Nadu
🌍 Country:   India
📍 Latitude:  11.0102
📍 Longitude: 76.9701
📊 Confidence: 2/3 APIs agreed
🔍 Source:     ipapi.co
